In [ ]:
from transformers import AutoTokenizer
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import torch
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re


**Question 11**

In [ ]:
import nltk
from nltk import word_tokenize, TweetTokenizer
nltk.download('punkt')
nltk.download('punkt_tab')
from transformers import AutoTokenizer

# Instantiate a tokenizer based on a pre-trained model

              
def tokenize_text(sample_text, model_name="bert-base-uncased"):
    """
    Tokenizes the input text using the specified transformer tokenizer.

    Args:
        sample_text (str): The text to be tokenized.
        model_name (str): The name of the pre-trained model's tokenizer. Default is "bert-base-uncased".

    Returns:
        dict: A dictionary containing tokenized output (input_ids, attention_mask, etc.).
    """
    # Load the tokenizer
    #tokenizer = tokenizer(sample_text) # Enter code here
    tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased",
    use_fast=True,              # Uses a fast C++-based tokenizer (usually recommended)
    add_prefix_space=False,     # Prepends a space to text (useful for models like GPT-2/RoBERTa)
    revision="main"             # Specify the exact branch of the model repository
)

    # Tokenize the text
    tokenized_output = tokenizer(
        sample_text,
        padding="max_length",   # Pads to the maximum length of the model
        truncation='only_second',        # Truncate to the maximum length allowed by the model
        max_length=512,         # Maximum sequence length
        return_tensors="pt"     # Return PyTorch tensors
    )

    return tokenized_output

sample_text = "Transformers are a state-of-the-art approach in natural language processing."
tokenized_data = tokenize_text(sample_text,
                            model_name="bert-base-uncased")
tokenized_data['input_ids']


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/uchegodfrey/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/uchegodfrey/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


tensor([[  101, 19081,  2024,  1037,  2110,  1011,  1997,  1011,  1996,  1011,
          2396,  3921,  1999,  3019,  2653,  6364,  1012,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

In [ ]:
# Training dataset
data = [
    {"text": "I love this product!", "label": 1},
    {"text": "This is the worst experience ever.", "label": 0},
    {"text": "Absolutely fantastic!", "label": 1},
    {"text": "Not worth the price.", "label": 0},
    {"text": "Great quality and service.", "label": 1},
    {"text": "I will never buy this again.", "label": 0},
]


**Question 12**

In [ ]:
texts = [item["text"] for item in data]
labels = [item["label"] for item in data]

# Step 2: Convert text data to numerical features using TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english') # Enter code here

# Step 3: Split the data into training and test sets
X_train, X_test, y_train, y_test =  train_test_split(texts, labels, test_size=0.2, random_state=42)  # Enter code here


X_train_tfidf = vectorizer.fit_transform(X_train)  # Enter code here
X_test_tfidf = vectorizer.transform(X_test) # Enter code here

# Step 4: Train a Logistic Regression model
model = LogisticRegression() # Enter code here
model.fit(X_train_tfidf, y_train)

# Step 5: Make predictions and evaluate the model
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")


Accuracy: 50.00%


**Question 13**

In [ ]:
def compute_tfidf(documents, max_features=1000):
    """
    Computes the TF-IDF scores for a given set of document texts.

    Args:
        documents (list of str): A list of document texts.
        max_features (int): Maximum number of features to include in the TF-IDF matrix.

    Returns:
        DataFrame: A pandas DataFrame where rows correspond to documents
                   and columns correspond to terms, with TF-IDF scores as values.
        TfidfVectorizer: The fitted TfidfVectorizer object.
    """
    # Step 1: Initialize the TfidfVectorizer
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english') # Enter code here

    # Step 2: Fit and transform the documents
    tfidf_matrix = vectorizer.fit_transform(documents)  # Enter code here

    # Step 3: Convert the result to a pandas DataFrame
    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=vectorizer.get_feature_names_out(),
        index=[f"Doc_{i+1}" for i in range(len(documents))]
    )

    return tfidf_df, vectorizer

# Example document texts
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "Never jump over the lazy dog quickly.",
    "A fox is quick and smart.",
]

# Compute TF-IDF scores
tfidf_df, vectorizer = compute_tfidf(documents, max_features=10)

# Display the TF-IDF DataFrame
print(tfidf_df["fox"])


Doc_1    0.366180
Doc_2    0.000000
Doc_3    0.517856
Name: fox, dtype: float64


**Question 14**

In [ ]:
# Step 1: Load a pre-trained sentiment classification pipeline
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

texts = [item["text"] for item in data]
true_labels = [item["label"] for item in data]

# Step 3: Generate predictions using the pre-trained model
predictions = []
for text in texts:
    result = classifier(text)  # Enter code here
    sentiment = result[0]['label']  # Enter code here
    if sentiment == 'POSITIVE':
        predictions.append(1)
    else:
        predictions.append(0)

# Step 4: Calculate accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(true_labels, predictions)
print(f"Accuracy: {accuracy:.2%}")


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6299.57it/s]


Accuracy: 100.00%


**Question 15**

In [ ]:

nltk.download('stopwords')
nltk.download('punkt')

class SimpleTokenizer:
    def __init__(self):
        self.vocab = {}  # Vocabulary to map token to ID
        self.inverse_vocab = {}  # Inverse mapping to get token from ID
        self.next_id = 1  # Start the IDs from 1 (0 reserved for padding, for example)
        self.stop_words = set(stopwords.words('english'))  # Set of stop words

    def tokenize(self, text):
        """
        Tokenizes the input text into words by splitting on spaces and handling punctuation.

        Args:
        - text (str): The input text to tokenize.

        Returns:
        - List of tokens (words)
        """
        # Remove unwanted characters and split on spaces
    
        tokens = word_tokenize(text.lower())  # Make the text lowercase and tokenize
        text = [t for t  in tokens if t not in self.stop_words]  # Enter code here  # Make the text lowercase
        tokens = [t for t in text if re.match(r'^[a-zA-Z]+$', t)]  # Enter code here  # Regex to match words
        return tokens

    def build_vocab(self, texts):
        """
        Build vocabulary from a list of texts.

        Args:
        - texts (list of str): List of text samples to build vocab from.
        """
        for text in texts:
            tokens = self.tokenize(text)  # Enter code here
            for token in tokens:
                if token not in self.vocab:
                    self.vocab[token] = self.next_id
                    self.inverse_vocab[self.next_id] = token
                    self.next_id += 1

    def convert_tokens_to_ids(self, tokens):
        """
        Converts a list of tokens into a list of IDs using the vocabulary.

        Args:
        - tokens (list of str): List of tokens to convert to IDs.

        Returns:
        - List of IDs corresponding to the tokens.
        """
        # 0 for unknown words

        return [self.vocab.get(token, 0) for token in tokens]  # Enter code here

#Example usage
tokenizer = SimpleTokenizer()

# Sample texts to build vocab
texts = ["Hello, this is a simple tokenizer.", "Let's tokenize this text!"]
tokenizer.build_vocab(texts)

# Convert text to tokens
tokens = tokenizer.tokenize("Hello, this is a simple tokenizer.")
print("Tokens:", tokens)

# Convert tokens to IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print("Token IDs:", token_ids)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/uchegodfrey/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/uchegodfrey/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Tokens: ['hello', 'simple', 'tokenizer']
Token IDs: [1, 2, 3]
